# Load marts into SQLite, then ask four questions

Put the cleaned CSVs into two databases and run the SQL in `sql/`. No charts. No join.

**Words used here**

- **Mart:** a cleaned analysis table. The CSVs in `data/processed/`.
- **SQLite:** a single-file database. We use it so the same SQL can run here and later in a SQL client.
- **Two files, two layers.** `database/cms.sqlite` holds CMS tables only. `database/uci.sqlite` holds the stay table only.
- **No attach / no join.** I will not open both files in one connection. A CMS star and a UCI `<30` rate are not the same number.
- **Peer group.** Compare like hospitals: type × ownership × Census region. Then look at CMS's own better / worse / no different label.
- **Eligible stay.** A UCI stay we can score for 30-day return. Not death, hospice, still-in, or invalid gender.
- **Rebuild.** If a processed CSV changes, re-run this notebook. The `.sqlite` files are not in git.

**Steps**

1. Confirm every processed CSV is on disk.
2. Load the CMS tables into `cms.sqlite`.
3. Load the UCI stay table into `uci.sqlite`.
4. Check row counts and keys against the CSVs.
5. Run the four queries in `sql/`, one layer at a time.

Processed files are not edited.


In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

In [2]:
def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "data" / "processed").is_dir():
            return path
    raise FileNotFoundError(
        "Could not find the project root. Run this notebook from the "
        "repository folder or from notebooks/."
    )

PROJECT_ROOT = find_project_root(Path.cwd())
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
DATABASE_DIR = PROJECT_ROOT / "database"
DATABASE_DIR.mkdir(parents=True, exist_ok=True)

CMS_DB_PATH = DATABASE_DIR / "cms.sqlite"
UCI_DB_PATH = DATABASE_DIR / "uci.sqlite"

print(PROJECT_ROOT)
print(PROCESSED_DATA_DIR.exists())
print(CMS_DB_PATH)
print(UCI_DB_PATH)

C:\Users\Micaela\Documents\CODING\hospital-operations-analytics
True
C:\Users\Micaela\Documents\CODING\hospital-operations-analytics\database\cms.sqlite
C:\Users\Micaela\Documents\CODING\hospital-operations-analytics\database\uci.sqlite


## Confirm the processed files

I am only loading what notebooks `02` and `03` already saved. Table names will match the CSV stems so later SQL can say `cms_facility_mart`, not a new nickname.

IDs stay text. `010001` has to stay `010001`. Same for `encounter_id`, `patient_nbr`, and `payer_code`.

In [3]:
CMS_TABLES = [
    ("cms_facility_profile", ["facility_id"]),
    ("cms_facility_mart", ["facility_id"]),
    ("cms_unplanned_mart", ["facility_id", "measure_id"]),
    ("cms_hcahps_mart", ["facility_id", "hcahps_measure_id"]),
    ("cms_timely_ed_mart", ["facility_id", "measure_id"]),
    ("cms_footnote_crosswalk", ["footnote"]),
    ("cms_measure_dates", ["measure_id"]),
]
UCI_TABLES = [
    ("uci_encounter_mart", ["encounter_id"]),
]

TEXT_COLS = {
    "cms_facility_profile": ["facility_id", "zip_code", "phone"],
    "cms_facility_mart": ["facility_id", "zip_code", "phone"],
    "cms_unplanned_mart": ["facility_id", "measure_id", "footnote"],
    "cms_hcahps_mart": ["facility_id", "hcahps_measure_id"],
    "cms_timely_ed_mart": ["facility_id", "measure_id", "footnote"],
    "cms_footnote_crosswalk": ["footnote"],
    "cms_measure_dates": ["measure_id"],
    "uci_encounter_mart": [
        "encounter_id",
        "patient_nbr",
        "payer_code",
        "diag_1",
        "diag_2",
        "diag_3",
    ],
}

csv_counts = {}
for table_name, _keys in CMS_TABLES + UCI_TABLES:
    path = PROCESSED_DATA_DIR / f"{table_name}.csv"
    if not path.exists():
        raise FileNotFoundError(path)
    dtype = {col: "string" for col in TEXT_COLS[table_name]}
    preview = pd.read_csv(path, dtype=dtype, nrows=0)
    n_rows = sum(1 for _ in path.open(encoding="utf-8")) - 1
    csv_counts[table_name] = n_rows
    print(f"{path.name}: {n_rows:,} rows, {len(preview.columns)} cols")

cms_facility_profile.csv: 5,419 rows, 24 cols
cms_facility_mart.csv: 5,419 rows, 74 cols
cms_unplanned_mart.csv: 67,060 rows, 13 cols
cms_hcahps_mart.csv: 23,950 rows, 11 cols
cms_timely_ed_mart.csv: 32,606 rows, 9 cols
cms_footnote_crosswalk.csv: 32 rows, 2 cols
cms_measure_dates.csv: 171 rows, 6 cols
uci_encounter_mart.csv: 101,766 rows, 60 cols


## Load CMS

One connection, one file. I replace the file if it already exists so a re-run does not stack duplicate tables.

After the load I check: same row count as the CSV, unique keys, and `facility_id` still six characters.

In [4]:
def read_mart(table_name: str) -> pd.DataFrame:
    path = PROCESSED_DATA_DIR / f"{table_name}.csv"
    dtype = {col: "string" for col in TEXT_COLS[table_name]}
    return pd.read_csv(path, dtype=dtype)


def load_tables(db_path: Path, tables: list[tuple[str, list[str]]]) -> None:
    if db_path.exists():
        db_path.unlink()
    with sqlite3.connect(db_path) as conn:
        for table_name, keys in tables:
            frame = read_mart(table_name)
            frame.to_sql(table_name, conn, index=False, if_exists="replace")
            n_db = pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {table_name}", conn)["n"].item()
            assert n_db == csv_counts[table_name], (table_name, n_db, csv_counts[table_name])
            if keys:
                key_sql = ", ".join(keys)
                n_dup = pd.read_sql_query(
                    f"""
                    SELECT COUNT(*) AS n FROM (
                        SELECT {key_sql}, COUNT(*) AS c
                        FROM {table_name}
                        GROUP BY {key_sql}
                        HAVING c > 1
                    )
                    """,
                    conn,
                )["n"].item()
                assert n_dup == 0, (table_name, keys, n_dup)
            print(f"{db_path.name} / {table_name}: {n_db:,} rows")


load_tables(CMS_DB_PATH, CMS_TABLES)

with sqlite3.connect(CMS_DB_PATH) as cms_conn:
    id_len = pd.read_sql_query(
        "SELECT MIN(LENGTH(facility_id)) AS mn, MAX(LENGTH(facility_id)) AS mx "
        "FROM cms_facility_mart",
        cms_conn,
    )
    sample_ids = pd.read_sql_query(
        "SELECT facility_id FROM cms_facility_mart ORDER BY facility_id LIMIT 3",
        cms_conn,
    )
    attached = pd.read_sql_query("PRAGMA database_list", cms_conn)

print("facility_id length min/max:", id_len.to_dict("records")[0])
print("sample facility_id:", sample_ids["facility_id"].tolist())
print("attached databases:", attached["name"].tolist())
assert set(attached["name"]) == {"main"}
assert (id_len["mn"] == 6).all() and (id_len["mx"] == 6).all()

cms.sqlite / cms_facility_profile: 5,419 rows
cms.sqlite / cms_facility_mart: 5,419 rows


cms.sqlite / cms_unplanned_mart: 67,060 rows
cms.sqlite / cms_hcahps_mart: 23,950 rows


cms.sqlite / cms_timely_ed_mart: 32,606 rows
cms.sqlite / cms_footnote_crosswalk: 32 rows
cms.sqlite / cms_measure_dates: 171 rows
facility_id length min/max: {'mn': 6, 'mx': 6}
sample facility_id: ['010001', '010005', '010006']
attached databases: ['main']


## Load UCI

A second connection, a second file. I do not pass the CMS connection in. One stay per `encounter_id`.

In [5]:
load_tables(UCI_DB_PATH, UCI_TABLES)

with sqlite3.connect(UCI_DB_PATH) as uci_conn:
    n_enc = pd.read_sql_query(
        "SELECT COUNT(*) AS n, COUNT(DISTINCT encounter_id) AS n_id FROM uci_encounter_mart",
        uci_conn,
    )
    n_eligible = pd.read_sql_query(
        "SELECT SUM(eligible_for_readmit) AS n_eligible FROM uci_encounter_mart",
        uci_conn,
    )["n_eligible"].item()
    attached = pd.read_sql_query("PRAGMA database_list", uci_conn)

print(n_enc.to_dict("records")[0])
print("eligible_for_readmit:", int(n_eligible))
print("attached databases:", attached["name"].tolist())
assert n_enc["n"].item() == n_enc["n_id"].item() == csv_counts["uci_encounter_mart"]
assert int(n_eligible) == 99337
assert set(attached["name"]) == {"main"}

uci.sqlite / uci_encounter_mart: 101,766 rows
{'n': 101766, 'n_id': 101766}
eligible_for_readmit: 99337
attached databases: ['main']


## Two files, still separate

Both files should exist. Each connection should only see `main`. I am not attaching `uci.sqlite` onto the CMS connection.

In [6]:
print("cms.sqlite exists:", CMS_DB_PATH.exists(), "size", CMS_DB_PATH.stat().st_size)
print("uci.sqlite exists:", UCI_DB_PATH.exists(), "size", UCI_DB_PATH.stat().st_size)

with sqlite3.connect(CMS_DB_PATH) as cms_conn:
    cms_tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name",
        cms_conn,
    )["name"].tolist()
    cms_attached = pd.read_sql_query("PRAGMA database_list", cms_conn)["name"].tolist()

with sqlite3.connect(UCI_DB_PATH) as uci_conn:
    uci_tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name",
        uci_conn,
    )["name"].tolist()
    uci_attached = pd.read_sql_query("PRAGMA database_list", uci_conn)["name"].tolist()

print("CMS tables:", cms_tables)
print("UCI tables:", uci_tables)
print("CMS attached:", cms_attached)
print("UCI attached:", uci_attached)

assert cms_tables == sorted(name for name, _ in CMS_TABLES)
assert uci_tables == sorted(name for name, _ in UCI_TABLES)
assert "uci_encounter_mart" not in cms_tables
assert "cms_facility_mart" not in uci_tables
assert cms_attached == ["main"]
assert uci_attached == ["main"]
print("ok: two files, no cross-layer tables")

cms.sqlite exists: True size 22847488
uci.sqlite exists: True size 26669056
CMS tables: ['cms_facility_mart', 'cms_facility_profile', 'cms_footnote_crosswalk', 'cms_hcahps_mart', 'cms_measure_dates', 'cms_timely_ed_mart', 'cms_unplanned_mart']
UCI tables: ['uci_encounter_mart']
CMS attached: ['main']
UCI attached: ['main']
ok: two files, no cross-layer tables


## Four SQL questions

The databases are loaded. Queries live in `sql/` so they can be reused later. I read each file and run it here.

- CMS: rank hospitals inside a peer group on `READM_30_HF` (`cms_peer_rank.sql`).
- CMS: stars and HF publication by state and ownership (`cms_state_ownership.sql`).
- UCI: crude `<30` rate by age, admission type, and prior-acute band (`uci_readmission_segments.sql`).
- UCI: the same rate by the locked 0 / 1 / 2+ prior-acute cut (`uci_utilization_cte.sql`).

Keep numerator and denominator. Missing is missing. Eligible stays only on UCI. No crosswalk.


In [7]:
from IPython.display import display

SQL_DIR = PROJECT_ROOT / "sql"

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 48)
pd.set_option("display.max_rows", 20)


def run_sql_file(db_path: Path, filename: str) -> pd.DataFrame:
    sql = (SQL_DIR / filename).read_text(encoding="utf-8")
    with sqlite3.connect(db_path) as conn:
        return pd.read_sql_query(sql, conn)


def rate_interval(n_readmit: int, n: int, z: float = 1.96) -> tuple[float, float, float]:
    """Normal interval for a crude rate. Report with N. Not a model CI."""
    p = n_readmit / n
    se = (p * (1 - p) / n) ** 0.5
    return p, max(0.0, p - z * se), min(1.0, p + z * se)


print("sql files:", sorted(p.name for p in SQL_DIR.glob("*.sql")))


sql files: ['cms_peer_rank.sql', 'cms_state_ownership.sql', 'uci_readmission_segments.sql', 'uci_utilization_cte.sql']


## CMS. Peer rank on heart-failure readmission

I am ranking published `READM_30_HF` scores inside type × ownership × region. Lower score is better. Rank is computed on every hospital with a score, not only the ones CMS already called worse.

A rank of 1 in a peer of 1 is not a finding. CMS's `compared_to_national` is the published flag. Peer rank is the extra question: among hospitals that look like this one, where does the score sit?


In [8]:
hf_rank = run_sql_file(CMS_DB_PATH, "cms_peer_rank.sql")
print(hf_rank.shape)
print(hf_rank["compared_to_national"].value_counts(dropna=False))
print()
print("published HF scores:", len(hf_rank))
print("peer groups:", hf_rank.groupby(["hospital_type", "hospital_ownership", "census_region"]).ngroups)
print("singleton peers:", (hf_rank["n_in_peer"] == 1).sum(), "hospitals")

hf_label = (
    hf_rank.groupby("compared_to_national", dropna=False)
    .agg(n_hospitals=("facility_id", "size"), mean_score=("score", "mean"), median_den=("denominator", "median"))
    .reset_index()
)
hf_label


(3253, 12)
compared_to_national
No different than the national rate    3194
Worse than the national rate             38
Better than the national rate            21
Name: count, dtype: int64

published HF scores: 3253
peer groups: 68
singleton peers: 10 hospitals


,compared_to_national,n_hospitals,mean_score,median_den
0,Better than the national rate,21,17.804762,555.0
1,No different than the national rate,3194,21.347307,289.0
2,Worse than the national rate,38,25.776316,426.0


In [9]:
worse = hf_rank.loc[
    hf_rank["compared_to_national"] == "Worse than the national rate"
].sort_values("score", ascending=False)

print("worse than national:", len(worse))
print("of those, n_in_peer >= 20:", (worse["n_in_peer"] >= 20).sum())
worse.loc[
    worse["n_in_peer"] >= 20,
    [
        "facility_id",
        "facility_name",
        "state",
        "census_region",
        "score",
        "denominator",
        "rank_in_peer",
        "n_in_peer",
    ],
].head(12)


worse than national: 38
of those, n_in_peer >= 20: 29


,facility_id,facility_name,state,census_region,score,denominator,rank_in_peer,n_in_peer
654,230017,BRONSON METHODIST HOSPITAL,MI,Midwest,28.4,1088.0,357,357
1300,330399,ST BARNABAS HOSPITAL,NY,Northeast,27.6,273.0,317,317
1299,330009,BRONXCARE HOSPITAL CENTER,NY,Northeast,27.5,762.0,316,317
653,360125,ASHTABULA COUNTY MEDICAL CENTER,OH,Midwest,26.9,332.0,356,357
3172,050030,OROVILLE HOSPITAL,CA,West,26.7,239.0,247,247
1397,04010F,VA CENTRAL AR. VETERANS HEALTHCARE SYSTEM LR,AR,South,26.4,627.0,46,46
1298,330226,UNITY HOSPITAL,NY,Northeast,26.0,1071.0,315,317
983,330056,BROOKLYN HOSPITAL CENTER - DOWNTOWN CAMPUS,NY,Northeast,25.7,253.0,35,35
652,360354,WEST CHESTER HOSPITAL,OH,Midwest,25.6,532.0,355,357
1545,110105,COLQUITT REGIONAL MEDICAL CENTER,GA,South,25.6,427.0,145,145


In [10]:
south_vol = hf_rank.loc[
    (hf_rank["hospital_type"] == "Acute Care Hospitals")
    & (hf_rank["hospital_ownership"] == "Voluntary non-profit - Private")
    & (hf_rank["census_region"] == "South")
].sort_values("rank_in_peer")

print("South, acute care, voluntary non-profit private:", len(south_vol))
print(south_vol["compared_to_national"].value_counts())
pd.concat([south_vol.head(3), south_vol.tail(3)])[
    ["facility_id", "facility_name", "state", "score", "compared_to_national", "denominator", "rank_in_peer", "n_in_peer"]
]


South, acute care, voluntary non-profit private: 474
compared_to_national
No different than the national rate    468
Better than the national rate            4
Worse than the national rate             2
Name: count, dtype: int64


,facility_id,facility_name,state,score,compared_to_national,denominator,rank_in_peer,n_in_peer
2018,490021,CENTRA HEALTH - LYNCHBURG GEN HOSPITAL,VA,17.3,Better than the national rate,1405.0,1,474
2019,440082,ASCENSION SAINT THOMAS HOSPITAL,TN,17.4,Better than the national rate,1157.0,2,474
2020,100177,CAPE CANAVERAL HOSPITAL,FL,18.1,Better than the national rate,343.0,3,474
2489,370025,SAINT FRANCIS HOSPITAL MUSKOGEE,OK,24.7,No different than the national rate,512.0,472,474
2491,510058,CAMDEN CLARK MEDICAL CENTER,WV,25.1,Worse than the national rate,754.0,473,474
2490,420070,PRISMA HEALTH TUOMEY HOSPITAL,SC,25.1,Worse than the national rate,703.0,473,474


**CMS peer-rank findings**

- 3,253 hospitals have a published HF readmission score. CMS calls 38 worse than national, 21 better, and 3,194 no different. Another 1,537 HF rows have no score (too few cases or not reported). Those stay out of the rank.
- Most of the 38 worse hospitals sit at the bottom of a large peer (for example Bronson Methodist, MI, 28.4, rank 357 of 357 Midwest voluntary non-profit private acute-care hospitals; St. Barnabas, NY, 27.6, 317 of 317 in the Northeast peer). That is the useful cut: CMS already flagged them, and they are also last among similar hospitals.
- Some worse labels sit in tiny peers (a single VA hospital, a peer of 7). Rank is not the story there. Use the CMS label and the denominator, not a 1-of-1 rank.
- In the largest Southern peer (474 voluntary non-profit private acute-care hospitals), the best published scores are about 17.3–17.4 (Lynchburg General, VA; Ascension Saint Thomas, TN). The two CMS-worse hospitals in that peer are Prisma Health Tuomey (SC) and Camden Clark (WV), both 25.1, tied at 473 of 474.
- This is not a UCI `<30` rate. I did not open `uci.sqlite` for these queries.


## CMS. State and ownership

Count hospitals. Count how many have a star. Average the star only among the rated. Same idea for HF: count published scores and CMS-worse flags. A state with few published stars is not "low quality." It is mostly unpublished.


In [11]:
state_own = run_sql_file(CMS_DB_PATH, "cms_state_ownership.sql")
print(state_own.shape)

with sqlite3.connect(CMS_DB_PATH) as cms_conn:
    ownership = pd.read_sql_query(
        """
        SELECT
          hospital_ownership,
          COUNT(*) AS n_hospitals,
          SUM(CASE WHEN overall_rating IS NOT NULL THEN 1 ELSE 0 END) AS n_with_star,
          AVG(overall_rating) AS mean_star_among_rated
        FROM cms_facility_profile
        GROUP BY hospital_ownership
        ORDER BY n_hospitals DESC
        """,
        cms_conn,
    )
    region = pd.read_sql_query(
        """
        SELECT
          census_region,
          COUNT(*) AS n_hospitals,
          SUM(CASE WHEN overall_rating IS NOT NULL THEN 1 ELSE 0 END) AS n_with_star,
          AVG(overall_rating) AS mean_star_among_rated
        FROM cms_facility_profile
        GROUP BY census_region
        ORDER BY n_hospitals DESC
        """,
        cms_conn,
    )
    unpublished = pd.read_sql_query(
        """
        SELECT
          state,
          COUNT(*) AS n_hospitals,
          SUM(CASE WHEN overall_rating IS NOT NULL THEN 1 ELSE 0 END) AS n_with_star,
          1.0 * SUM(CASE WHEN overall_rating IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*) AS pct_rated,
          AVG(overall_rating) AS mean_star_among_rated
        FROM cms_facility_profile
        GROUP BY state
        HAVING COUNT(*) >= 20
        ORDER BY pct_rated ASC
        """,
        cms_conn,
    )

print("ownership")
display(ownership)
print("region")
display(region)
print("states with the lowest share of published stars (n >= 20 hospitals)")
display(unpublished.head(8))
print("state x ownership rows with at least 20 hospitals")
state_own.loc[state_own["n_hospitals"] >= 20].sort_values("n_hospitals", ascending=False).head(12)


(436, 7)
ownership


,hospital_ownership,n_hospitals,n_with_star,mean_star_among_rated
0,Voluntary non-profit - Private,2322,1619,3.307597
1,Proprietary,1063,502,2.790837
2,Government - Hospital District or Authority,512,228,2.916667
3,Government - Local,393,179,2.921788
4,Voluntary non-profit - Other,353,246,3.304878
5,Voluntary non-profit - Church,264,218,3.380734
6,Government - State,209,38,2.868421
7,Veterans Health Administration,132,112,4.160714
8,Physician,80,18,3.666667
9,Government - Federal,42,11,2.818182


region


,census_region,n_hospitals,n_with_star,mean_star_among_rated
0,South,2072,1186,3.111298
1,Midwest,1537,857,3.353559
2,West,1076,643,3.239502
3,Northeast,669,478,3.169456
4,Territory,65,10,1.700000


states with the lowest share of published stars (n >= 20 hospitals)


,state,n_hospitals,n_with_star,pct_rated,mean_star_among_rated
0,PR,59,7,0.118644,1.714286
1,SD,61,18,0.295082,3.888889
2,MT,63,20,0.317460,3.250000
3,ND,47,16,0.340426,3.250000
4,LA,161,59,0.366460,2.813559
5,NE,93,36,0.387097,3.500000
6,KS,139,55,0.395683,3.181818
7,AK,25,10,0.400000,2.900000


state x ownership rows with at least 20 hospitals


,state,hospital_ownership,n_hospitals,n_with_star,mean_star_among_rated,n_with_hf_score,n_hf_worse
372,TX,Proprietary,171,73,3.013699,74,0
38,CA,Voluntary non-profit - Private,147,120,3.416667,119,2
373,TX,Voluntary non-profit - Private,134,91,3.703297,96,0
294,NY,Voluntary non-profit - Private,125,101,2.702970,101,4
330,PA,Voluntary non-profit - Private,120,106,3.471698,108,0
302,OH,Voluntary non-profit - Private,103,76,3.710526,79,3
39,CA,Proprietary,98,64,2.406250,62,3
412,WI,Voluntary non-profit - Private,97,69,3.840580,79,0
124,IL,Voluntary non-profit - Private,94,68,3.102941,66,0
76,FL,Proprietary,84,64,2.390625,65,0


**CMS state / ownership findings**

- 5,419 hospitals. 3,174 have a star (mean 3.21 among the rated). 2,245 have no star. That gap has to stay in every average.
- Proprietary hospitals: 1,063 facilities, only 502 rated, mean star 2.79. Voluntary non-profit private: 2,322 facilities, 1,619 rated, mean 3.31. DoD hospitals: 32 facilities, 0 stars. Veterans Health Administration: 132 facilities, 112 rated, mean 4.16. Different reporting rules. Do not rank ownership types as if they all publish the same way.
- Puerto Rico: 59 hospitals, 7 stars (12% rated), mean 1.71 among those seven. South Dakota, Montana, North Dakota, Louisiana also publish stars on well under half of hospitals. A state map without `n_with_star` is misleading.
- Among states with at least 20 published stars, Utah (4.24) and Colorado (3.96) sit high. That is still an unadjusted geographic view, not a peer comparison.
- Next notebook (`05`) can take these cuts into charts. This pass only locks the counts.


## UCI. Segment rates

Crude `<30` share among eligible stays. `age` is already a band in the file. Prior acute is inpatient + ED in the year before the stay (0 / 1 / 2+). Cells with fewer than 50 stays are dropped.

I also print the one-way cuts (age, admission type, diagnosis group) so a tiny three-way cell does not become the story.


In [12]:
seg = run_sql_file(UCI_DB_PATH, "uci_readmission_segments.sql")
print("three-way cells with n >= 50:", len(seg))

with sqlite3.connect(UCI_DB_PATH) as uci_conn:
    overall = pd.read_sql_query(
        """
        SELECT
          COUNT(*) AS n_encounters,
          SUM(readmit_30) AS n_readmit
        FROM uci_encounter_mart
        WHERE eligible_for_readmit = 1
        """,
        uci_conn,
    )
    by_age = pd.read_sql_query(
        """
        SELECT age, SUM(readmit_30) AS n_readmit, COUNT(*) AS n_encounters,
               1.0 * SUM(readmit_30) / COUNT(*) AS readmit_30_rate
        FROM uci_encounter_mart
        WHERE eligible_for_readmit = 1
        GROUP BY age
        ORDER BY age
        """,
        uci_conn,
    )
    by_adm = pd.read_sql_query(
        """
        SELECT admission_type, SUM(readmit_30) AS n_readmit, COUNT(*) AS n_encounters,
               1.0 * SUM(readmit_30) / COUNT(*) AS readmit_30_rate
        FROM uci_encounter_mart
        WHERE eligible_for_readmit = 1
        GROUP BY admission_type
        ORDER BY n_encounters DESC
        """,
        uci_conn,
    )
    by_dx = pd.read_sql_query(
        """
        SELECT diag_1_group, SUM(readmit_30) AS n_readmit, COUNT(*) AS n_encounters,
               1.0 * SUM(readmit_30) / COUNT(*) AS readmit_30_rate
        FROM uci_encounter_mart
        WHERE eligible_for_readmit = 1
        GROUP BY diag_1_group
        HAVING COUNT(*) >= 50
        ORDER BY n_encounters DESC
        """,
        uci_conn,
    )

n, k = int(overall["n_encounters"].item()), int(overall["n_readmit"].item())
p, lo, hi = rate_interval(k, n)
print(f"eligible overall: {k:,} / {n:,} = {p:.3f} (95% interval {lo:.3f} to {hi:.3f})")
print()
print("by age")
display(by_age)
print("by admission type")
display(by_adm)
print("by primary diagnosis group (n >= 50)")
display(by_dx)
print("highest three-way cells")
seg.head(10)


three-way cells with n >= 50:

 95


eligible overall: 11,312 / 99,337 = 0.114 (95% interval 0.112 to 0.116)

by age


,age,n_readmit,n_encounters,readmit_30_rate
0,[0-10),3,160,0.018750
1,[10-20),40,690,0.057971
2,[20-30),236,1649,0.143117
3,[30-40),424,3764,0.112646
4,[40-50),1024,9607,0.106589
5,[50-60),1667,17060,0.097714
6,[60-70),2493,22058,0.113020
7,[70-80),3052,25327,0.120504
8,[80-90),2065,16434,0.125654
9,[90-100),308,2588,0.119011


by admission type


,admission_type,n_readmit,n_encounters,readmit_30_rate
0,Emergency,6194,52368,0.118278
1,Elective,1958,18667,0.104891
2,Urgent,2056,18130,0.113403
3,None,1103,10144,0.108734
4,Trauma Center,0,18,0.000000
5,Newborn,1,10,0.100000


by primary diagnosis group (n >= 50)


,diag_1_group,n_readmit,n_encounters,readmit_30_rate
0,circulatory,3459,29583,0.116925
1,endocrine,1461,11304,0.129246
2,respiratory,1111,9918,0.112019
3,digestive,961,9072,0.105930
4,symptoms,683,7601,0.089857
5,injury,850,6851,0.124069
6,genitourinary,546,4963,0.110014
7,musculoskeletal,471,4935,0.095441
8,neoplasm,341,3131,0.108911
9,infectious,315,2553,0.123384


highest three-way cells


,age,admission_type,prior_acute_band,n_readmit,n_encounters,readmit_30_rate
0,[20-30),Urgent,2+,30,76,0.394737
1,[20-30),Emergency,2+,99,316,0.313291
2,[30-40),None,2+,24,77,0.311688
3,[30-40),Elective,2+,36,119,0.302521
4,[30-40),Urgent,2+,36,125,0.288000
5,[80-90),Elective,2+,104,395,0.263291
6,[40-50),Urgent,2+,97,391,0.248082
7,[50-60),Elective,2+,103,472,0.218220
8,[30-40),Emergency,2+,112,519,0.215800
9,[40-50),Emergency,2+,246,1164,0.211340


**UCI segment findings**

- Eligible overall: 11,312 returns in 99,337 stays, **11.4%** (about 11.2% to 11.6%). The 2,429 ineligible stays stay out. Using all 101,766 stays would mix people who cannot return into the denominator.
- Age is not a straight climb. [20-30) is 14.3% (236 / 1,649). [50-60) is 9.8%. [70-80) is 12.1% and [80-90) is 12.6%. [0-10) is 1.9% on only 160 stays; I will not lead with it.
- Emergency admissions: 11.8% (6,194 / 52,368). Elective: 10.5% (1,958 / 18,667). Urgent sits in between. Trauma Center and Newborn have N under 20. Do not quote those rates.
- Circulatory is the largest diagnosis group (29,583 stays, 11.7%). Endocrine is 12.9% (1,461 / 11,304). Symptoms is lower (9.0%). These are crude chapter rates, not risk-adjusted.
- The three-way table is dominated by **prior acute 2+**. Young adults with 2+ prior acute visits sit near 30%+ in the top cells. That is the same signal as the next query. I am not treating a CMS HF score as this rate.


## UCI. Prior utilization

The locked cut is 0 / 1 / 2+ prior acute visits (inpatient + ED). The CTE is only a readable wrapper. Same eligibility rule.


In [13]:
util = run_sql_file(UCI_DB_PATH, "uci_utilization_cte.sql")
util = util.copy()
intervals = [rate_interval(int(r.n_readmit), int(r.n_encounters)) for r in util.itertuples()]
util["rate_lo"] = [x[1] for x in intervals]
util["rate_hi"] = [x[2] for x in intervals]
print(util)

with sqlite3.connect(UCI_DB_PATH) as uci_conn:
    cross = pd.read_sql_query(
        """
        SELECT
          inpatient_band,
          emergency_band,
          SUM(readmit_30) AS n_readmit,
          COUNT(*) AS n_encounters,
          1.0 * SUM(readmit_30) / COUNT(*) AS readmit_30_rate
        FROM uci_encounter_mart
        WHERE eligible_for_readmit = 1
        GROUP BY inpatient_band, emergency_band
        ORDER BY inpatient_band, emergency_band
        """,
        uci_conn,
    )
cross


  prior_acute_band  n_readmit  n_encounters  readmit_30_rate   rate_lo   rate_hi
0                0       5215         61736         0.084473  0.082279  0.086666
1                1       2424         19618         0.123560  0.118955  0.128165
2               2+       3673         17983         0.204248  0.198356  0.210141


,inpatient_band,emergency_band,n_readmit,n_encounters,readmit_30_rate
0,0,0,5215,61736,0.084473
1,0,1,348,3465,0.100433
2,0,2+,126,1039,0.121270
3,1,0,2076,16153,0.128521
4,1,1,290,2000,0.145000
5,1,2+,149,830,0.179518
6,2+,0,2138,10354,0.206490
7,2+,1,459,2009,0.228472
8,2+,2+,511,1751,0.291833


**UCI utilization findings**

- No prior acute visit: **8.4%** (5,215 / 61,736). One: **12.4%** (2,424 / 19,618). Two or more: **20.4%** (3,673 / 17,983). The step is large and the Ns are not small.
- Splitting inpatient and ED does not change the direction. Zero and zero is 8.4%. Two-plus inpatient and two-plus ED is 29.2% (511 / 1,751). Prior inpatient alone (2+, no ED) is already 20.6%.
- This is association in a 1999–2008 diabetes extract. It does not say that cutting ED visits would cut readmission. It does say prior acute use is the strongest operational cut this file can show.
- Next notebook (`06`) can chart these bands with intervals. The model (`07`) should treat prior utilization as a candidate feature, known before discharge.

SQL pass is done. Two databases. Four questions. No join.
